In [13]:
import numpy as np
from scipy.stats import norm
from dataclasses import dataclass

@dataclass
class GammaScalpingResult:
    futures_to_trade: float
    updated_delta_hedge: float

# Black-Scholes formula for calculating d1
def calculate_d1(underlying_price: float, strike_price: float, time_to_expiration: float, risk_free_rate: float, volatility: float) -> float:
    d1 = (np.log(underlying_price / strike_price) + (risk_free_rate + 0.5 * volatility ** 2) * time_to_expiration) / (volatility * np.sqrt(time_to_expiration))
    return d1

# Use d1 to calculate delta and gamma
def calculate_greeks_from_d1(d1: float, underlying_price: float, time_to_expiration: float, volatility: float) -> tuple[float, float]:
    delta_call = norm.cdf(d1)
    delta_put = delta_call - 1
    gamma = norm.pdf(d1) / (underlying_price * volatility * np.sqrt(time_to_expiration))
    return delta_call, delta_put, gamma

# Gamma scalping logic that outputs a single dataclass, with injected state
def gamma_scalping_signal(total_option_delta: float, current_delta_hedge: float) -> GammaScalpingResult | None:
    # Calculate the new hedge
    new_delta_hedge = -total_option_delta
    futures_to_trade = new_delta_hedge - current_delta_hedge  # Positive means buy, negative means sell

    # If there's no need to trade, return None
    if futures_to_trade == 0:
        return None

    # Return a dataclass with the trading signal and the updated delta hedge
    return GammaScalpingResult(futures_to_trade=futures_to_trade, updated_delta_hedge=new_delta_hedge)

# Simulate underlying prices using Geometric Brownian Motion (GBM)
def simulate_gbm(initial_price: float, drift: float, volatility: float, time_horizon: float, num_steps: int) -> list[float]:
    time_step = time_horizon / num_steps  # Time step
    simulated_prices = [initial_price]
    for _ in range(num_steps):
        random_shock = np.random.normal(0, 1)  # Generate random shock
        new_price = simulated_prices[-1] * np.exp((drift - 0.5 * volatility ** 2) * time_step + volatility * np.sqrt(time_step) * random_shock)
        simulated_prices.append(new_price)
    return simulated_prices

# Example usage with real-time price data:
if __name__ == "__main__":
    # Parameters
    initial_underlying_price: float = 100.0  # Initial underlying price
    strike_price: float = 100.0   # Strike price of the options (ATM)
    time_horizon: float = 1.0     # Time period for simulation (1 year)
    risk_free_rate: float = 0.01  # Risk-free rate
    implied_volatility: float = 0.2  # Implied volatility
    drift_rate: float = 0.05    # Drift (expected return)

    num_simulation_steps: int = 100  # Number of steps in the simulation

    # Simulate underlying price using GBM
    simulated_prices = simulate_gbm(initial_underlying_price, drift_rate, implied_volatility, time_horizon, num_simulation_steps)

    # Initial state for the delta hedge
    d1_initial = calculate_d1(initial_underlying_price, strike_price, time_horizon, risk_free_rate, implied_volatility)
    delta_call, delta_put, _ = calculate_greeks_from_d1(d1_initial, initial_underlying_price, time_horizon, implied_volatility)
    current_delta_hedge: float = -(delta_call + delta_put)  # Initial delta hedge

    # Simulate gamma scalping with signals and injected state
    for price in simulated_prices:
        # Calculate d1 for the current price
        d1_current = calculate_d1(price, strike_price, time_horizon, risk_free_rate, implied_volatility)
        
        # Calculate the total option delta using d1
        delta_call, delta_put, _ = calculate_greeks_from_d1(d1_current, price, time_horizon, implied_volatility)
        total_option_delta: float = delta_call * 1.0 + delta_put * 1.0

        # Pass the calculated total_option_delta into gamma_scalping_signal
        result = gamma_scalping_signal(total_option_delta, current_delta_hedge)
        
        if result is not None:
            print(f"Price: {price:.2f}, Futures to Trade: {result.futures_to_trade:.4f}")
            # Update the delta hedge state
            current_delta_hedge = result.updated_delta_hedge


Price: 102.87, Futures to Trade: -0.1100
Price: 104.82, Futures to Trade: -0.0708
Price: 104.98, Futures to Trade: -0.0055
Price: 103.66, Futures to Trade: 0.0472
Price: 105.15, Futures to Trade: -0.0532
Price: 106.01, Futures to Trade: -0.0298
Price: 107.96, Futures to Trade: -0.0645
Price: 109.37, Futures to Trade: -0.0442
Price: 111.72, Futures to Trade: -0.0687
Price: 110.57, Futures to Trade: 0.0330
Price: 113.55, Futures to Trade: -0.0821
Price: 114.04, Futures to Trade: -0.0124
Price: 118.91, Futures to Trade: -0.1101
Price: 117.31, Futures to Trade: 0.0332
Price: 121.49, Futures to Trade: -0.0817
Price: 122.01, Futures to Trade: -0.0089
Price: 120.47, Futures to Trade: 0.0273
Price: 119.35, Futures to Trade: 0.0214
Price: 118.62, Futures to Trade: 0.0145
Price: 119.67, Futures to Trade: -0.0207
Price: 120.96, Futures to Trade: -0.0241
Price: 119.62, Futures to Trade: 0.0251
Price: 119.59, Futures to Trade: 0.0004
Price: 124.87, Futures to Trade: -0.0887
Price: 129.29, Futures t